---
title: 'Lab 5: Potoki i kroswalidacja'
subtitle: Biblioteki Python w analizie danych
author: Tomasz Rodak
toc-title: Spis treści
jupyter: python3
---

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rodakt/BPwAD/blob/v2/laby/lab_5.ipynb)

Na lab 4 zbudowaliśmy własne estymatory zgodne z API scikit-learn. Teraz zobaczymy, po co to było: prawidłowo zaimplementowane estymatory i transformatory można łączyć w potoki (*pipelines*), walidować krzyżowo i automatycznie optymalizować hiperparametry.

W tym arkuszu przeprowadzimy kompletną analizę klasyfikacyjną na zbiorze Titanic — od wczytania danych z cechami mieszanymi (numeryczne i kategoryczne) i brakami, przez budowę potoku, aż po przeszukiwanie hiperparametrów za pomocą `GridSearchCV` i `RandomizedSearchCV`.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

## 1. Dane

Zbiór Titanic zawiera informacje o pasażerach statku RMS Titanic. Zadaniem jest przewidzenie, czy pasażer przeżył katastrofę (`survived`), na podstawie cech takich jak klasa biletu, płeć, wiek czy opłata za bilet.


### 1.1 Wczytanie zbioru

Wczytaj zbiór Titanic z OpenML za pomocą `fetch_openml`. Funkcja zwraca obiekt z atrybutami `data` (DataFrame z cechami) i `target` (Series z etykietami).


In [ ]:
from sklearn.datasets import fetch_openml

titanic = fetch_openml('titanic', version=1, as_frame=True, parser='auto')
df = titanic.data
y = titanic.target

### 1.2 Eksploracja

Zapoznaj się z danymi:

1. Wyświetl kilka pierwszych wierszy (`df.head()`).
2. Sprawdź typy kolumn (`df.dtypes`).
3. Policz braki w każdej kolumnie (`df.isna().sum()`).
4. Ile jest wierszy i kolumn?


### 1.3 Wybór cech

Część kolumn jest mało przydatna do modelowania (np. `name`, `ticket`, `cabin`, `boat`, `body`, `home.dest`). Wybierzemy podzbiór cech, z którym będziemy dalej pracować:


In [ ]:
feature_cols = ['pclass', 'sex', 'age', 'sibsp', 'parch', 'fare', 'embarked']
X = df[feature_cols]
X.head()

Zidentyfikuj, które z wybranych kolumn są numeryczne, a które kategoryczne. Wypisz ich nazwy — będą potrzebne w sekcji 3.

*Wskazówka:* `X.select_dtypes(include='number').columns` zwraca nazwy kolumn numerycznych, a `X.select_dtypes(include='category').columns` — kategorycznych. Jeśli kolumny nie mają odpowiednich typów, sprawdź `X.dtypes` i zdecyduj ręcznie na podstawie zawartości.


### 1.4 Podział na zbiór treningowy i testowy


In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"Treningowy: {X_train.shape}, testowy: {X_test.shape}")

## 2. Prosty potok — cechy numeryczne

Na początek zbudujemy potok korzystający wyłącznie z cech numerycznych. Pozwoli to zrozumieć mechanikę `Pipeline`, zanim dodamy `ColumnTransformer`.


### 2.1 Budowa potoku

Wybierz z `X_train` i `X_test` tylko kolumny numeryczne. Zbuduj potok z trzema krokami:

1. `SimpleImputer(strategy='median')` — wypełnienie braków medianą.
2. `StandardScaler()` — standaryzacja cech.
3. `LogisticRegression()` — klasyfikator.


In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression


### 2.2 Trening i ocena

Dopasuj potok do danych treningowych (kolumny numeryczne) i oblicz predykcje na zbiorze testowym. Wyświetl macierz pomyłek i dokładność (*accuracy*).


In [ ]:
from sklearn.metrics import confusion_matrix, accuracy_score


### 2.3 Kroswalidacja

Zamiast jednego podziału train/test, użyj kroswalidacji. Uruchom `cross_val_score` z potokiem na *całym* zbiorze (kolumny numeryczne). Użyj 5-krotnej kroswalidacji.


In [ ]:
from sklearn.model_selection import cross_val_score


Zwróć uwagę, że `cross_val_score` przyjmuje *potok*, a nie sam klasyfikator. Dzięki temu w każdym foldzie `SimpleImputer` i `StandardScaler` są dopasowywane wyłącznie do danych treningowych tego folda — dane walidacyjne nie wpływają na obliczone mediany ani parametry skalowania.


## 3. `ColumnTransformer` — cechy mieszane

Zbiór Titanic zawiera kolumny kategoryczne (`sex`, `embarked`) obok numerycznych. Nie można ich przekazać wprost do `StandardScaler` ani do `LogisticRegression`. Potrzebujemy mechanizmu, który zastosuje różne transformacje do różnych grup kolumn — to rola `ColumnTransformer`.


### 3.1 Budowa preprocesora

`ColumnTransformer` przyjmuje listę trójek `(nazwa, transformator, kolumny)`. Każda grupa kolumn przechodzi przez swój potok transformacji, a wyniki są łączone w jedną macierz.

Zbuduj preprocesor z dwiema ścieżkami:

- **Numeryczna** (`age`, `sibsp`, `parch`, `fare`): `SimpleImputer(strategy='median')` → `StandardScaler()`.
- **Kategoryczna** (`pclass`, `sex`, `embarked`): `SimpleImputer(strategy='most_frequent')` → `OneHotEncoder(handle_unknown='ignore')`.

*Uwaga:* `pclass` ma wartości 1, 2, 3 — wygląda jak liczba, ale reprezentuje klasę biletu (porządkową/kategoryczną). Traktowanie jej jako cechy kategorycznej (one-hot encoding) jest w tym kontekście uzasadnione.


In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

num_cols = [...]  # uzupełnij
cat_cols = [...]  # uzupełnij

num_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

cat_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer([
    ('num', num_pipeline, num_cols),
    ('cat', cat_pipeline, cat_cols)
])

### 3.2 Pełny potok

Połącz preprocesor z klasyfikatorem w jeden potok:


In [ ]:
pipe = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(max_iter=1000))
])

### 3.3 Trening i ocena

Dopasuj potok do `X_train`, `y_train` (tym razem wszystkie wybrane kolumny, nie tylko numeryczne). Oblicz predykcje na `X_test`. Wyświetl macierz pomyłek i dokładność.

Porównaj wynik z sekcją 2 — czy dodanie cech kategorycznych poprawiło klasyfikację?


### 3.4 Kroswalidacja

Uruchom 5-krotną kroswalidację na pełnym potoku. Porównaj średnią dokładność z wynikiem z sekcji 2.3.


### 3.5 Inspekcja potoku

Po dopasowaniu potoku można zajrzeć do jego wnętrza. Spróbuj odpowiedzieć na poniższe pytania, odwołując się do atrybutów dopasowanego potoku.

1. Jakie mediany obliczył `SimpleImputer` w ścieżce numerycznej?
2. Jakie kategorie wykrył `OneHotEncoder`?
3. Ile cech ma macierz po transformacji?

*Wskazówka:* Do kroków potoku można się dostać za pomocą składni słownikowej (`pipe['nazwa_kroku']`) lub atrybutu `named_steps`.


## 4. `GridSearchCV`

Potok z sekcji 3 używa domyślnych hiperparametrów. `GridSearchCV` pozwala przeszukać wszystkie kombinacje wartości z podanej siatki i wybrać najlepszą konfigurację na podstawie kroswalidacji.


### 4.1 Definiowanie siatki

Hiperparametry kroków potoku adresuje się w formacie `nazwa_kroku__nazwa_parametru` (podwójne podkreślenie). Dla kroków zagnieżdżonych (potok wewnątrz `ColumnTransformer`) łańcuch się wydłuża, np. `preprocessor__num__imputer__strategy` oznacza: krok `preprocessor` → transformator `num` → krok `imputer` → parametr `strategy`.

Zdefiniuj siatkę przeszukiwania:


In [ ]:
param_grid = {
    'preprocessor__num__imputer__strategy': ['mean', 'median'],
    'classifier__C': [0.01, 0.1, 1, 10],
    'classifier__penalty': ['l1', 'l2'],
    'classifier__solver': ['liblinear']
}

Ile kombinacji zawiera ta siatka? Ile łącznie dopasowań modelu wykona `GridSearchCV` z 5-krotną kroswalidacją?


### 4.2 Przeszukiwanie

Uruchom `GridSearchCV` na potoku z sekcji 3.2. Użyj `scoring='accuracy'` i `cv=5`.


In [ ]:
from sklearn.model_selection import GridSearchCV


### 4.3 Analiza wyników

1. Wyświetl najlepsze hiperparametry (`search.best_params_`) i najlepszy wynik kroswalidacji (`search.best_score_`).
2. Oblicz dokładność najlepszego modelu na zbiorze testowym (`search.best_estimator_.score(X_test, y_test)` lub `search.score(X_test, y_test)`).
3. Wyświetl macierz pomyłek dla predykcji na zbiorze testowym.


### 4.4 Tabela wyników

Atrybut `cv_results_` zawiera szczegółowe wyniki wszystkich kombinacji. Przekształć go w DataFrame i wyświetl kilka najlepszych konfiguracji posortowanych według `rank_test_score`.

*Wskazówka:* `pd.DataFrame(search.cv_results_)` tworzy DataFrame ze wszystkimi wynikami. Przydatne kolumny: `params`, `mean_test_score`, `std_test_score`, `rank_test_score`.


## 5. `RandomizedSearchCV`

`GridSearchCV` przeszukuje *wszystkie* kombinacje — przy wielu hiperparametrach i szerokich zakresach wartości liczba kombinacji rośnie wykładniczo. `RandomizedSearchCV` losuje zadaną liczbę kombinacji z podanych rozkładów, co pozwala przeszukać znacznie większą przestrzeń w tym samym budżecie obliczeniowym.


### 5.1 Definiowanie rozkładów

Zamiast list wartości podajemy rozkłady prawdopodobieństwa. Biblioteka `scipy.stats` dostarcza rozkłady ciągłe i dyskretne:

- `uniform(loc, scale)` — rozkład jednostajny na przedziale $[\text{loc},\; \text{loc} + \text{scale}]$.
- `loguniform(a, b)` — rozkład log-jednostajny na przedziale $[a, b]$; $\log(X)$ jest jednostajny. Odpowiedni dla parametrów, które działają w różnych rzędach wielkości (np. `C`).
- `randint(low, high)` — rozkład jednostajny na liczbach całkowitych z przedziału $[\text{low}, \text{high})$.

Zdefiniuj rozkłady przeszukiwania:


In [ ]:
from scipy.stats import loguniform

param_distributions = {
    'preprocessor__num__imputer__strategy': ['mean', 'median'],
    'classifier__C': loguniform(0.001, 100),
    'classifier__penalty': ['l1', 'l2'],
    'classifier__solver': ['liblinear']
}

Zwróć uwagę na różnicę: `classifier__C` w `GridSearchCV` to lista konkretnych wartości (4 elementy), a tutaj to rozkład ciągły — `RandomizedSearchCV` będzie z niego losować.


### 5.2 Przeszukiwanie

Uruchom `RandomizedSearchCV` z `n_iter=40` (40 losowych kombinacji), `cv=5`, `scoring='accuracy'`, `random_state=42`.


In [ ]:
from sklearn.model_selection import RandomizedSearchCV


### 5.3 Analiza wyników

1. Wyświetl najlepsze hiperparametry i najlepszy wynik kroswalidacji.
2. Oblicz dokładność na zbiorze testowym.
3. Porównaj z wynikami `GridSearchCV` z sekcji 4. Czy `RandomizedSearchCV` znalazł lepszą konfigurację?


### 5.4 Porównanie strategii

Uzupełnij tabelę:

|  | `GridSearchCV` | `RandomizedSearchCV` |
|---|---|---|
| Liczba sprawdzonych kombinacji | ... | ... |
| Łączna liczba dopasowań (kombinacje × foldy) | ... | ... |
| Najlepszy wynik kroswalidacji | ... | ... |
| Dokładność na zbiorze testowym | ... | ... |
| Najlepsze `C` | ... | ... |

Który z tych dwóch algorytmów jest lepiej przystosowany do przeszukiwania dużych przestrzeni hiperparametrów? Dlaczego?


## 6. Zadanie dodatkowe: porównanie klasyfikatorów

Jedną z zalet potokowej architektury jest łatwość podmiany klasyfikatora — preprocesor pozostaje ten sam, zmienia się tylko ostatni krok.


### 6.1 Kandydaci

Porównaj co najmniej trzy klasyfikatory z tym samym preprocesorem z sekcji 3.1. Dla każdego z nich uruchom 5-krotną kroswalidację i zapisz średnią dokładność. Propozycje:

- `LogisticRegression(max_iter=1000)`
- `KNeighborsClassifier(n_neighbors=5)`
- `RandomForestClassifier(n_estimators=100, random_state=42)`

Aby podmienić klasyfikator w istniejącym potoku, użyj `pipe.set_params(classifier=...)` lub utwórz nowy potok z tym samym preprocesorem.


In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier


### 6.2 Tabela wyników

Zebrany wyniki w tabeli: klasyfikator, średnia dokładność z kroswalidacji, odchylenie standardowe. Który model działa najlepiej na tym zbiorze?


### 6.3 Serializacja

Zapisz najlepszy potok do pliku za pomocą `joblib.dump()` i wczytaj go ponownie. Sprawdź, że wczytany potok daje te same predykcje.


In [ ]:
from joblib import dump, load
